# Tiny Aya extraction: second training seed and enum repair

1. Repair invalid closed-field values (for example `issue_category`) in the round-2 adapter's outputs by letting the model choose among the allowed values (the public adapter and its predictions are downloaded from Hugging Face).
2. Train the same recipe again with another seed (7) and evaluate it on the v2 and v1 test sets.
3. Repair the new adapter's outputs the same way.

Needs a GPU, internet and the `HF_TOKEN` secret. About 2.2 hours on a T4.

In [ ]:
!pip install -q aligntune peft bitsandbytes accelerate jsonschema huggingface_hub 2>&1 | tail -2

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download
token = UserSecretsClient().get_secret('HF_TOKEN').strip().strip('"').strip("'")
os.environ['HF_TOKEN'] = token
login(token=token)  # also saved on disk for the training and evaluation subprocesses
print('logged in as', HfApi().whoami(token=token)['name'])
snapshot_download('iamvisheshsrivastava/lexsi-extraction-data-v2', repo_type='dataset', local_dir='/kaggle/working/data/final', token=token)
print(os.listdir('/kaggle/working/data/final'))

In [ ]:
%%writefile /kaggle/working/schema.py
"""The extraction target: one flat JSON object per support ticket or invoice."""

DOC_TYPES = ["support_ticket", "invoice"]
LANGUAGES = {"en": "English", "de": "German", "fr": "French", "hi": "Hinglish (Hindi written in Latin script mixed with English)"}
CATEGORIES = ["billing", "shipping", "technical", "account", "refund", "other"]
PRIORITIES = ["low", "medium", "high", "urgent"]

FIELDS = ["doc_type", "language", "party_name", "reference_id", "product", "issue_category", "priority",
          "amount", "currency", "date_iso"]

SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": FIELDS,
    "properties": {
        "doc_type": {"enum": DOC_TYPES},
        "language": {"enum": list(LANGUAGES)},
        "party_name": {"type": ["string", "null"]},
        "reference_id": {"type": ["string", "null"]},
        "product": {"type": ["string", "null"]},
        "issue_category": {"enum": CATEGORIES},
        "priority": {"enum": PRIORITIES},
        "amount": {"type": ["number", "null"]},
        "currency": {"type": ["string", "null"], "pattern": "^[A-Z]{3}$"},
        "date_iso": {"type": ["string", "null"], "pattern": "^\\d{4}-\\d{2}-\\d{2}$"},
    },
}

INSTRUCTION = (
    "Extract the information from the document below into a single JSON object with exactly these keys: "
    + ", ".join(FIELDS) + ". Use null for anything not stated, except doc_type, language, issue_category and priority, "
    "which must always be one of the allowed values and never null. "
    "doc_type is support_ticket or invoice; language is en, de, fr or hi (hi = Hinglish); "
    "issue_category is one of " + "/".join(CATEGORIES) + "; priority is one of " + "/".join(PRIORITIES) + "; "
    "amount is a number; currency is a 3-letter ISO code; date_iso is YYYY-MM-DD. Output only the JSON."
)


In [ ]:
%%writefile /kaggle/working/eval_extraction.py
"""Metrics for structured-extraction outputs: JSON validity, schema pass, field F1 and exact match.

Predictions and references are dicts keyed by example id. A prediction is the raw model text; the
reference is the gold JSON object. Usage as a module: evaluate(predictions, references, schema).
"""
import json
import re

from jsonschema import Draft202012Validator


def parse_json(text):
    """Return the first JSON object in text, or None."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        start = text.find("{")
        while start != -1:
            depth = 0
            for end in range(start, len(text)):
                depth += text[end] == "{"
                depth -= text[end] == "}"
                if depth == 0:
                    try:
                        value = json.loads(text[start:end + 1])
                        return value if isinstance(value, dict) else None
                    except json.JSONDecodeError:
                        break
            start = text.find("{", start + 1)
    return None


def normalise(value):
    if isinstance(value, str):
        return re.sub(r"\s+", " ", value).strip().casefold()
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value


def field_scores(prediction, reference):
    """Return (true positives, predicted fields, reference fields) over top-level key/value pairs."""
    gold = {k: normalise(v) for k, v in reference.items() if v not in (None, "", [])}
    pred = {k: normalise(v) for k, v in (prediction or {}).items() if v not in (None, "", [])}
    tp = sum(1 for k, v in pred.items() if gold.get(k) == v)
    return tp, len(pred), len(gold)


def evaluate(predictions, references, schema):
    validator = Draft202012Validator(schema)
    n = len(references)
    valid_json = schema_pass = exact = 0
    tp = n_pred = n_gold = 0
    for key, reference in references.items():
        parsed = parse_json(predictions.get(key, ""))
        valid_json += parsed is not None
        if parsed is not None and not list(validator.iter_errors(parsed)):
            schema_pass += 1
        a, b, c = field_scores(parsed, reference)
        tp, n_pred, n_gold = tp + a, n_pred + b, n_gold + c
        exact += parsed is not None and a == b == c
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gold if n_gold else 0.0
    return {
        "n": n,
        "valid_json_rate": valid_json / n,
        "schema_pass_rate": schema_pass / n,
        "field_precision": precision,
        "field_recall": recall,
        "field_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "exact_match_rate": exact / n,
    }


In [ ]:
%%writefile /kaggle/working/eval_model.py
"""Evaluate a model (optionally with a LoRA adapter) on the extraction test set.

Reports valid-JSON rate, schema pass rate, field F1, exact match (overall and per language / doc type)
plus generation throughput (for the ROI comparison). Meant for a GPU session (Kaggle / Colab).

Run: python eval_model.py --model CohereLabs/tiny-aya-global --data data/final --out results/baseline.json
     python eval_model.py --model CohereLabs/tiny-aya-global --adapter out/adapter --out results/tuned.json
"""
import argparse
import json
import time
from collections import defaultdict
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from eval_extraction import evaluate
from schema import INSTRUCTION, SCHEMA


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--adapter")
    parser.add_argument("--data", default="data/final")
    parser.add_argument("--split", default="test")
    parser.add_argument("--out", required=True)
    parser.add_argument("--limit", type=int)
    parser.add_argument("--batch", type=int, default=8)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--no-stop", action="store_true", help="do not stop at the closing brace (the first evaluation run did not)")
    parser.add_argument("--prompt", choices=["short", "full"], default="short",
                        help="short = the minimal prompt used for training; full = the full schema instruction")
    args = parser.parse_args()

    rows = [json.loads(line) for line in open(Path(args.data) / f"{args.split}.jsonl", encoding="utf-8")]
    rows = rows[: args.limit] if args.limit else rows
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model, device_map="auto", torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type="nf4"))
    if args.adapter:
        model = PeftModel.from_pretrained(model, args.adapter)
    model.eval()

    # Stop when the model ends its turn, ends its response, or closes the (flat) JSON object. Without this a few
    # outputs ran to the token limit and held up their whole batch in the first evaluation run.
    end_response = tokenizer.convert_tokens_to_ids("<|END_RESPONSE|>")
    eos_ids = sorted({i for i in (tokenizer.eos_token_id, end_response) if isinstance(i, int) and i != tokenizer.unk_token_id})
    gen_kwargs = dict(max_new_tokens=args.max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids)
    if not args.no_stop:
        gen_kwargs.update(stop_strings=["}"], tokenizer=tokenizer)
    predictions, generated_tokens, at_limit, start = {}, 0, 0, time.time()
    for i in range(0, len(rows), args.batch):
        chunk = rows[i:i + args.batch]
        texts = [tokenizer.apply_chat_template([{"role": "user", "content": r["prompt"] if args.prompt == "short" else
                                                 INSTRUCTION + "\n\nDocument:\n" + r["prompt"].split("Document:\n", 1)[1]}], tokenize=False,
                                               add_generation_prompt=True) for r in chunk]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            try:
                output = model.generate(**inputs, **gen_kwargs)
            except (TypeError, ValueError) as error:  # stop_strings unsupported by this transformers version: fall back
                if "stop_strings" not in gen_kwargs:
                    raise
                print("warning: stop_strings unavailable, generating without it:", str(error)[:120], flush=True)
                gen_kwargs.pop("stop_strings")
                gen_kwargs.pop("tokenizer")
                output = model.generate(**inputs, **gen_kwargs)
        for j, row in enumerate(chunk):
            new = output[j][inputs["input_ids"].shape[1]:]
            at_limit += int((new != tokenizer.pad_token_id).sum() >= args.max_new_tokens)
            generated_tokens += int((new != tokenizer.pad_token_id).sum())
            predictions[i + j] = tokenizer.decode(new, skip_special_tokens=True)
        print(f"[eval] {min(i + args.batch, len(rows))}/{len(rows)} done, {time.time() - start:.0f}s elapsed", flush=True)
    seconds = time.time() - start

    references = {i: json.loads(r["output"]) for i, r in enumerate(rows)}
    result = {"model": args.model, "adapter": args.adapter, "split": args.split, "prompt": args.prompt,
              "stop_at_brace": not args.no_stop, "outputs_at_token_limit": at_limit, "n_documents": len(rows),
              "overall": evaluate(predictions, references, SCHEMA),
              "tokens_per_second": generated_tokens / seconds, "seconds": seconds}
    for field in ("language", "doc_type"):
        groups = defaultdict(list)
        for i, r in enumerate(rows):
            groups[r[field]].append(i)
        result["by_" + field] = {g: evaluate({i: predictions[i] for i in idx}, {i: references[i] for i in idx}, SCHEMA)
                                 for g, idx in groups.items()}
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    Path(args.out).with_suffix(".predictions.json").write_text(json.dumps(predictions, indent=1, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(result["overall"], indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/train_sft.py
"""LoRA supervised fine-tuning of a Tiny Aya model on the curated extraction data with AlignTune.

Meant for a GPU session (Kaggle / Colab). Needs `pip install aligntune` and a Hugging Face token that has
accepted the model's license.

Run: python train_sft.py --model CohereLabs/tiny-aya-global --data data/final/train.jsonl --out out
"""
import argparse

from aligntune.core.backend_factory import create_sft_trainer


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--data", default="data/final/train.jsonl")
    parser.add_argument("--out", default="out")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--seed", type=int, default=42, help="42 is the AlignTune default and was used for every run reported so far")
    args = parser.parse_args()
    trainer = create_sft_trainer(
        model_name=args.model,
        dataset_name=args.data,
        backend="trl",
        output_dir=args.out,
        num_epochs=args.epochs,
        batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=1024,
        use_peft=True,
        lora_r=16,
        lora_alpha=32,
        quantization={"load_in_4bit": True},
        task_type="sft",
        format_type="alpaca",
        seed=args.seed,
    )
    trainer.train()
    print("saved to", trainer.save_model(args.out + "/adapter"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repair_enums.py
"""Repair invalid enum values in model outputs by letting the model choose among the allowed values.

The fine-tuned model sometimes writes a value that is not allowed for a closed field (for example issue_category "error" or
"delivery"). This script finds such values in the generated text and replaces each with the allowed value that the SAME model rates
most likely at that position (the log-likelihood of the candidate, given the document prompt and the text generated before it). It
is constrained choice at the failing field only: no rules fitted to test documents, no changes to values that are already allowed, and
no help for missing keys or malformed JSON (those are counted, not fixed).

Run (GPU): python repair_enums.py --model CohereLabs/tiny-aya-global --adapter out/adapter --data data/final_v2 \
               --predictions results/v2_tuned.predictions.json --out results/v2_tuned_repaired.json
"""
import argparse
import json
import re
from pathlib import Path

from schema import CATEGORIES, DOC_TYPES, LANGUAGES, PRIORITIES

ENUMS = {"doc_type": DOC_TYPES, "language": list(LANGUAGES), "issue_category": CATEGORIES, "priority": PRIORITIES}


def find_invalid_enums(text):
    """Return [(key, start, end, current, quoted)] for closed fields whose value is not allowed.

    A quoted value that is not allowed (start/end delimit the text between the quotes) and a bare null (start/end delimit `null`)
    are both reported. Missing keys and numbers are not; use `count_unfixable` for those.
    """
    found = []
    for key, allowed in ENUMS.items():
        match = re.search(r'"%s"\s*:\s*(?:"([^"]*)"|(null))' % re.escape(key), text)
        if not match:
            continue
        if match.group(1) is not None and match.group(1) not in allowed:
            found.append((key, match.start(1), match.end(1), match.group(1), True))
        elif match.group(2) is not None:
            found.append((key, match.start(2), match.end(2), "null", False))
    return sorted(found, key=lambda item: item[1])


def count_unfixable(text):
    """Closed fields that are missing altogether; these are not repaired."""
    return sum(1 for key in ENUMS if not re.search(r'"%s"\s*:' % re.escape(key), text))


def repair_text(text, choose):
    """Replace each invalid enum value with choose(prefix, key, allowed); prefix is the text before the value, ending with an open quote.

    If choose returns None the value is left as it was.
    """
    changes, repaired, shift = [], text, 0
    for key, start, end, current, quoted in find_invalid_enums(text):
        prefix = repaired[: start + shift] + ("" if quoted else '"')
        best = choose(prefix, key, ENUMS[key])
        if best is None:
            continue
        replacement = best if quoted else '"' + best + '"'
        repaired = repaired[: start + shift] + replacement + repaired[end + shift:]
        shift += len(replacement) - (end - start)
        changes.append({"key": key, "from": current, "to": best})
    return repaired, changes


def main():
    import time

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    from eval_extraction import evaluate
    from schema import SCHEMA

    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--adapter")
    parser.add_argument("--data", required=True)
    parser.add_argument("--split", default="test")
    parser.add_argument("--predictions", required=True)
    parser.add_argument("--prompt", choices=["short", "full"], default="short")
    parser.add_argument("--out", required=True)
    args = parser.parse_args()

    rows = [json.loads(line) for line in open(Path(args.data) / f"{args.split}.jsonl", encoding="utf-8")]
    predictions = json.loads(Path(args.predictions).read_text(encoding="utf-8"))
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(
        args.model, device_map="auto", torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))
    if args.adapter:
        model = PeftModel.from_pretrained(model, args.adapter)
    model.eval()

    def prompt_text(row):
        from schema import INSTRUCTION
        content = row["prompt"] if args.prompt == "short" else INSTRUCTION + "\n\nDocument:\n" + row["prompt"].split("Document:\n", 1)[1]
        return tokenizer.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)

    start_token = "<|START_RESPONSE|>"
    probe = tokenizer.apply_chat_template([{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}], tokenize=False)
    uses_start = start_token in probe  # the model's replies begin with this token, which decoding with skip_special_tokens drops

    def make_chooser(row):
        context = prompt_text(row)
        if uses_start and not context.endswith(start_token):
            context += start_token
        n_context = tokenizer(context, add_special_tokens=False, return_tensors="pt").input_ids.shape[1]

        def choose(prefix, key, allowed):
            scores = []
            for candidate in allowed:
                ids = tokenizer(context + prefix + candidate + '"', return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
                with torch.no_grad():
                    logits = model(ids).logits[0, n_context - 1:-1].float().log_softmax(-1)
                # log-likelihood of everything after the prompt: the text before the field is the same for every candidate,
                # so the differences between candidates are the likelihood of the candidate value and its closing quote
                scores.append(logits.gather(1, ids[0, n_context:].unsqueeze(1)).sum().item())
            return allowed[max(range(len(allowed)), key=lambda i: scores[i])]
        return choose

    repaired, log, unfixable, start = {}, [], 0, time.time()
    for i, row in enumerate(rows):
        text = predictions.get(str(i), "")
        chooser = make_chooser(row)
        new_text, changes = repair_text(text, chooser)
        repaired[str(i)] = new_text
        unfixable += count_unfixable(text) > 0
        log += [dict(c, doc=i) for c in changes]
    references = {i: json.loads(r["output"]) for i, r in enumerate(rows)}
    before = evaluate({i: predictions.get(str(i), "") for i in range(len(rows))}, references, SCHEMA)
    after = evaluate({i: repaired[str(i)] for i in range(len(rows))}, references, SCHEMA)
    result = {"model": args.model, "adapter": args.adapter, "split": args.split, "documents": len(rows),
              "values_repaired": len(log), "documents_with_missing_closed_field": unfixable, "seconds": time.time() - start,
              "before": before, "after": after, "changes": log}
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    Path(args.out).with_suffix(".predictions.json").write_text(json.dumps(repaired, indent=1, ensure_ascii=False), encoding="utf-8")
    print(json.dumps({"values_repaired": len(log), "before": before, "after": after}, indent=2))


if __name__ == "__main__":
    main()


## 1. Enum repair on the round-2 adapter (seed 42)

In [ ]:
%cd /kaggle/working
from huggingface_hub import snapshot_download
snapshot_download('iamvisheshsrivastava/lexsi-tiny-aya-extraction-lora', local_dir='seed42', token=token, allow_patterns=['adapter_*', 'chat_template.jinja', 'tokenizer*', 'results/v2_tinyaya/v2_tuned.predictions.json'])
import os; print(os.listdir('seed42'))
!python repair_enums.py --model CohereLabs/tiny-aya-global --adapter seed42 --data data/final --predictions seed42/results/v2_tinyaya/v2_tuned.predictions.json --out results/seed42_repaired.json || echo 'repair failed, continuing'

## 2. Second training seed (7), same recipe

In [ ]:
!python train_sft.py --model CohereLabs/tiny-aya-global --data data/final/train.jsonl --out out_seed7 --epochs 2 --seed 7

In [ ]:
%cd /kaggle/working
!python eval_model.py --model CohereLabs/tiny-aya-global --adapter out_seed7/adapter --data data/final --split test --out results/seed7_tuned.json
!python eval_model.py --model CohereLabs/tiny-aya-global --adapter out_seed7/adapter --data data/final --split test_v1 --out results/seed7_v1test_tuned.json

## 3. Enum repair on the seed-7 adapter

In [ ]:
!python repair_enums.py --model CohereLabs/tiny-aya-global --adapter out_seed7/adapter --data data/final --predictions results/seed7_tuned.predictions.json --out results/seed7_repaired.json || echo 'repair failed'

In [ ]:
import json, glob
for path in sorted(glob.glob('results/*.json')):
    if path.endswith('predictions.json'):
        continue
    d = json.load(open(path))
    print(path, {k: round(v, 3) for k, v in (d.get('overall') or d.get('after') or {}).items() if isinstance(v, float)}, 'repaired:', d.get('values_repaired'))